# 05 MCP 双向体检

**测什么**: ① `law-search` server 的三个工具是否注册; ② stdio 传输能否完成 MCP 握手
(历史上此处曾无超时挂死); ③ streamable-http 传输能否起服务并被 client 拉到工具;
④ server 不在线时 agent 侧挂载是否优雅降级。

**判定**: 握手与拉工具成功为 PASS。stdio 与 http 是两条独立传输, 一条失败不影响另一条结论。
所有探针都带超时, 不会挂死 notebook。

**前置**: 无。端口 9381 被占用时自动改用空闲端口, 不会踩掉已跑着的 server。


In [ ]:
import asyncio, os, sys
from pathlib import Path

for cand in (Path.cwd(), *Path.cwd().parents):
    if (cand / "nbkit.py").is_file():
        NB_DIR = cand
        break
    if (cand / "tests_ipynb" / "nbkit.py").is_file():
        NB_DIR = cand / "tests_ipynb"
        break
else:
    raise RuntimeError("未找到 nbkit.py")

sys.path.insert(0, str(NB_DIR))

from nbkit import Checks, bootstrap

ROOT = bootstrap()
checks = Checks("05 MCP 双向体检")

print("解释器  :", sys.executable)
print("仓库根  :", ROOT)
print("HF_HOME :", os.getenv("HF_HOME", "(未设置)"))


In [ ]:
import os, sys, time
from pathlib import Path

from lawApp_LangGraph.config import settings as s
from lawApp_LangGraph.mcp.mcp_server import mcp
from nbkit import free_port, tcp_open, wait_port

names = sorted(t.name for t in mcp._tool_manager.list_tools())
checks.expect(
    names == ["recall_memory", "search_cases", "search_laws"],
    "law-search 三工具已注册",
    ok_detail=", ".join(names),
    fail_detail=f"注册结果与预期不符: {names}",
)

## 1. stdio 传输握手(带 90 秒超时)

子进程必须显式传 `HF_HOME`: MCP SDK 默认只继承一份白名单环境变量, 丢失 HF_HOME 后
子进程会去联网找 BGE 模型, 表现为长时间挂起而非报错(见 `tests/test_mcp.py` 注释)。

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

STDIO_TIMEOUT = 90.0
child_env = {k: os.environ.get(k, "") for k in ("PATH", "SYSTEMROOT", "USERPROFILE", "TEMP")}
child_env.update({"HF_HUB_OFFLINE": "1", "PYTHONIOENCODING": "utf-8"})
for k in ("HF_HOME", "HF_ENDPOINT", "HF_HUB_CACHE"):
    if os.environ.get(k):
        child_env[k] = os.environ[k]

# 用 -c 直接在子进程里 run(stdio), 不落地临时入口文件
stdio_args = [
    "-c",
    "from lawApp_LangGraph.mcp.mcp_server import mcp\nmcp.run(transport='stdio')\n",
]
params = StdioServerParameters(
    command=sys.executable, args=stdio_args, cwd=str(ROOT), env=child_env
)
errlog = ROOT / "tests_ipynb" / "_stdio_stderr.log"

async def probe_stdio():
    """拉子进程走一遍 initialize + list_tools, 返回工具名列表。"""
    with open(errlog, "w", encoding="utf-8") as err:
        async with stdio_client(params, errlog=err) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                res = await session.list_tools()
                return sorted(t.name for t in res.tools)

t0 = time.time()
try:
    stdio_names = await asyncio.wait_for(probe_stdio(), STDIO_TIMEOUT)
except asyncio.TimeoutError:
    checks.fail(
        "stdio 握手",
        f"超过 {STDIO_TIMEOUT:.0f}s 未完成 → 子进程 stderr 见 {errlog.name}; "
        "该问题即 tests/test_mcp.py::test_stdio_end_to_end 挂起的同一现象",
    )
except Exception as e:
    checks.fail("stdio 握手", f"{type(e).__name__}: {str(e)[:200]}")
else:
    checks.expect(
        stdio_names == ["recall_memory", "search_cases", "search_laws"],
        "stdio 握手 + tools/list",
        ok_detail=f"{time.time() - t0:.1f}s | {', '.join(stdio_names)}",
        fail_detail=f"工具列表不符: {stdio_names}",
    )

## 2. stdio 子进程 stderr(诊断用)

In [ ]:
if errlog.is_file():
    text = errlog.read_text(encoding="utf-8", errors="replace").strip()
    print(text[:1500] if text else "(stderr 为空)")
else:
    print("(未产生 stderr 文件)")

## 3. streamable-http 传输(起子进程 → 端口就绪 → client 拉工具 → 收工)

In [ ]:
import subprocess

port = s.mcp_port if not tcp_open(s.mcp_host, s.mcp_port, timeout=0.5) else free_port()
http_env = {**os.environ, "MCP_PORT": str(port), "MCP_HOST": "127.0.0.1"}
proc = subprocess.Popen(
    [sys.executable, "-m", "lawApp_LangGraph.mcp.mcp_server"],
    cwd=str(ROOT), env=http_env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8",
)
print(f"子进程 pid={proc.pid} 端口={port}")

async def probe_http(url):
    """用 adapters 的显式会话 API 连 http server 并列工具。"""
    from langchain_mcp_adapters.client import MultiServerMCPClient

    client = MultiServerMCPClient(
        {"law-search": {"url": url, "transport": "streamable_http"}}
    )
    async with client.session("law-search") as session:
        res = await session.list_tools()
        return sorted(t.name for t in res.tools)

ok_port = wait_port("127.0.0.1", port, timeout=30)
checks.expect(ok_port, "http 服务端口就绪", f"127.0.0.1:{port}", f"30s 内未监听(查看下方子进程输出)")

if ok_port:
    try:
        http_names = await asyncio.wait_for(probe_http(f"http://127.0.0.1:{port}/mcp"), 60)
    except Exception as e:
        checks.fail("streamable-http 挂载", f"{type(e).__name__}: {str(e)[:200]}")
    else:
        checks.expect(
            http_names == ["recall_memory", "search_cases", "search_laws"],
            "streamable-http 挂载 + tools/list",
            ok_detail=", ".join(http_names),
            fail_detail=f"工具列表不符: {http_names}",
        )
else:
    checks.skip("streamable-http 挂载", "端口未就绪")

proc.terminate()
try:
    out, _ = proc.communicate(timeout=15)
except subprocess.TimeoutExpired:
    proc.kill()
    out, _ = proc.communicate()
print("\n子进程输出:")
print((out or "").strip()[:1200] or "(空)")

## 4. agent 侧挂载: server 不在线时优雅降级

In [ ]:
import lawApp_LangGraph.mcp.mcp_client as mc

dead = f"http://127.0.0.1:{free_port()}/mcp"
mc.settings.mcp_server_url = dead
mc._loaded, mc._mcp_tools, mc._client = False, [], None
t0 = time.time()
tools = await asyncio.wait_for(mc.get_mcp_tools(), 60)
checks.expect(
    tools == [],
    "server 不在线 → 挂载降级为空",
    ok_detail=f"{time.time() - t0:.1f}s | url={dead}",
    fail_detail=f"预期空列表, 实得 {tools}",
)

## 5. adapters 挂载 stdio(复现 `tests/test_mcp.py` 的路径)

上面第 1 节走的是裸协议(`mcp.client.stdio` + `ClientSession`), 而 pytest 里挂死的那条路
走的是 `langchain-mcp-adapters` 的 `MultiServerMCPClient.get_tools()`。两者同用一个 stdio 入口,
故本项单独隔离: 若裸协议通而 adapters 超时, 说明问题在 adapters 的 stdio 取工具路径, 而非服务端。

In [ ]:
ADAPTER_TIMEOUT = 120.0

async def probe_adapters_stdio():
    """用 adapters 的 get_tools() 挂 stdio server(与 tests/test_mcp.py 同路径)。"""
    from langchain_mcp_adapters.client import MultiServerMCPClient

    client = MultiServerMCPClient(
        {
            "law-search-stdio": {
                "transport": "stdio",
                "command": sys.executable,
                "args": stdio_args,
                "cwd": str(ROOT),
                "env": child_env,
            }
        }
    )
    tools = await client.get_tools()
    return sorted(t.name for t in tools)

t0 = time.time()
try:
    adapter_names = await asyncio.wait_for(probe_adapters_stdio(), ADAPTER_TIMEOUT)
except asyncio.TimeoutError:
    checks.fail(
        "adapters get_tools(stdio)",
        f"超过 {ADAPTER_TIMEOUT:.0f}s 未返回 → 与 tests/test_mcp.py::test_stdio_end_to_end 挂起同源; "
        "裸协议可用, 故问题在 adapters 侧(参见 stderr 文件)",
    )
except Exception as e:
    checks.fail("adapters get_tools(stdio)", f"{type(e).__name__}: {str(e)[:200]}")
else:
    checks.expect(
        adapter_names == ["recall_memory", "search_cases", "search_laws"],
        "adapters get_tools(stdio)",
        ok_detail=f"{time.time() - t0:.1f}s | {', '.join(adapter_names)}",
        fail_detail=f"工具列表不符: {adapter_names}",
    )

## 汇总

In [ ]:
print(checks.report())